# 01 — N02 Candidate Manifest Integrity Audit

**안전 상태:** 이 노트북은 원본 CSV를 바꾸지 않습니다. 감사 보고서와 대기열만 생성합니다. 자동 검사에 통과해도 `FROZEN`으로 승인하지 않습니다.

## 실행 전 준비

로컬에서 ZIP을 압축 해제한 뒤 `PhysLLM_N02_Antigravity_Workspace` 폴더를 Google Drive의 `My Drive`로 업로드합니다. 가장 편한 방식은 Google Drive for desktop으로 폴더를 동기화한 뒤, 그 동기화 폴더를 Antigravity IDE에서 직접 여는 것입니다.

`MyDrive/PhysLLM_N02_Antigravity_Workspace/`

Colab 확장으로 원격 실행할 때 로컬 Antigravity 경로는 원격 커널에서 보이지 않을 수 있으므로 아래는 Google Drive 경로를 사용합니다. 동기화를 사용하지 않는다면 Python 스크립트를 수정할 때 최신 파일을 Drive에 다시 업로드해야 합니다.


In [1]:
# Colab에서는 Drive를 연결합니다. 일반 Jupyter/로컬 런타임에서는 건너뜁니다.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("google.colab 모듈이 없습니다. 일반 Jupyter 런타임입니다.")
except Exception as exc:
    print("Drive mount가 완료되지 않았습니다:", repr(exc))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from pathlib import Path
import os, sys, json, subprocess

PROJECT_ROOT = Path('/content/drive/MyDrive/PhysLLM_N02_Antigravity_Workspace')
MANIFEST = PROJECT_ROOT / 'data/candidate/N02_Main_Evaluation_CANDIDATE_Manifest_v1.0.3.csv'
AUDIT_SCRIPT = PROJECT_ROOT / 'scripts/audit_manifest.py'
REVIEW_LEDGER = PROJECT_ROOT / 'data/review/human_review_queue.csv'
OUT_DIR = PROJECT_ROOT / 'reports/audits/colab_run'

print('PROJECT_ROOT exists:', PROJECT_ROOT.exists(), PROJECT_ROOT)
print('MANIFEST exists:', MANIFEST.exists(), MANIFEST)
print('AUDIT_SCRIPT exists:', AUDIT_SCRIPT.exists(), AUDIT_SCRIPT)
if not MANIFEST.exists() or not AUDIT_SCRIPT.exists():
    raise FileNotFoundError(
        'Workspace files are missing from Drive. Upload this ZIP to Google Drive and extract it into '
        'MyDrive/PhysLLM_N02_Antigravity_Workspace, then rerun this cell.'
    )


PROJECT_ROOT exists: True /content/drive/MyDrive/PhysLLM_N02_Antigravity_Workspace
MANIFEST exists: True /content/drive/MyDrive/PhysLLM_N02_Antigravity_Workspace/data/candidate/N02_Main_Evaluation_CANDIDATE_Manifest_v1.0.3.csv
AUDIT_SCRIPT exists: True /content/drive/MyDrive/PhysLLM_N02_Antigravity_Workspace/scripts/audit_manifest.py


In [4]:
# Audit is CPU-oriented; this does not load GPT-2, Pythia or any large language model.
cmd = [sys.executable, str(AUDIT_SCRIPT), '--manifest', str(MANIFEST), '--out-dir', str(OUT_DIR)]
if REVIEW_LEDGER.exists():
    cmd += ['--review-ledger', str(REVIEW_LEDGER)]
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-4000:])
print('Return code:', result.returncode)
if not (OUT_DIR / 'audit_report.json').exists():
    raise RuntimeError('Audit report was not created; inspect the captured error output.')


STATUS: CANDIDATE_NEEDS_REVIEW
MANIFEST_SHA256: 4d2a7590b2c13310aa798fd89bb2a7f07c3fa26696967c0fcfc025b236b63158
ROWS: 400 | TOPICS: 100 | REVIEW_REQUIRED: 400
FROZEN: False
REPORT: /content/drive/MyDrive/PhysLLM_N02_Antigravity_Workspace/reports/audits/colab_run/audit_report.json

Return code: 0


In [5]:
import pandas as pd
report = json.loads((OUT_DIR / 'audit_report.json').read_text(encoding='utf-8'))
print(json.dumps({k: report.get(k) for k in [
    'status','manifest_filename','manifest_sha256','rows','unique_topics','styles',
    'code_rows','normalized_code_template_count','normalized_code_template_groups_repeated_3plus',
    'max_template_repetition','human_review_complete','review_required_rows','frozen_for_confirmatory_analysis'
]}, indent=2, ensure_ascii=False))
findings = pd.read_csv(OUT_DIR / 'findings.csv')
display(findings)


{
  "status": "CANDIDATE_NEEDS_REVIEW",
  "manifest_filename": "N02_Main_Evaluation_CANDIDATE_Manifest_v1.0.3.csv",
  "manifest_sha256": "4d2a7590b2c13310aa798fd89bb2a7f07c3fa26696967c0fcfc025b236b63158",
  "rows": 400,
  "unique_topics": 100,
  "styles": {
    "general_prose": 100,
    "question_dialogue": 100,
    "academic": 100,
    "code_plus_natural_language": 100
  },
  "code_rows": 100,
  "normalized_code_template_count": 31,
  "normalized_code_template_groups_repeated_3plus": 19,
  "max_template_repetition": 7,
  "human_review_complete": false,
  "review_required_rows": 400,
  "frozen_for_confirmatory_analysis": false
}


,severity,check,text_id,message
0,INFO,required_columns,NaN,All required columns are present.
1,INFO,no_missing_required_values,NaN,Rows with missing required values=0; empty tex...
2,INFO,expected_row_count,NaN,Observed 400 rows; expected 400 for this candi...
3,INFO,expected_topic_clusters,NaN,Observed 100 unique topic_id values; expected ...
4,INFO,four_rows_per_cluster,NaN,Clusters with row count !=4: 0.
5,INFO,style_balance,NaN,"Observed style counts={'general_prose': 100, '..."
6,INFO,unique_text_id,NaN,Rows involved in duplicate text_id values=0.
7,INFO,no_exact_duplicate_text,NaN,Rows involved in exact duplicate texts=0.
8,INFO,no_normalized_duplicate_text,NaN,Rows involved in whitespace/case-normalized du...
9,INFO,cluster_metadata_consistency,NaN,Topic metadata is constant within each topic_i...


## 선택 단계 — 독립 토크나이저 재측정

기본 감사의 토큰 수는 CSV에 저장된 값을 검사합니다. 더 강한 검증이 필요하면 아래 옵션을 `True`로 바꿔 Hugging Face 토크나이저로 각 문항을 다시 셉니다. 모델 가중치는 받지 않고 tokenizer 파일만 가져오며, 네트워크 연결이 필요합니다. 토크나이저 버전/특수 토큰 처리에 따른 차이를 숨기지 말고 보고서에 남깁니다.


In [6]:
RUN_INDEPENDENT_RETOKENIZATION = False  # True로 바꾸면 tokenizer를 독립 다운로드/재검사합니다.
if RUN_INDEPENDENT_RETOKENIZATION:
    cmd = [sys.executable, str(AUDIT_SCRIPT), '--manifest', str(MANIFEST),
           '--out-dir', str(OUT_DIR / 'retokenized'), '--retokenize']
    if REVIEW_LEDGER.exists():
        cmd += ['--review-ledger', str(REVIEW_LEDGER)]
    result = subprocess.run(cmd, text=True, capture_output=True)
    print(result.stdout)
    print(result.stderr[-4000:] if result.stderr else '')
    print('Return code:', result.returncode)
else:
    print('Skipped. Set RUN_INDEPENDENT_RETOKENIZATION=True to run the independent tokenizer pass.')


Skipped. Set RUN_INDEPENDENT_RETOKENIZATION=True to run the independent tokenizer pass.


## 해석 규칙

- `CANDIDATE_NEEDS_REVIEW`: 기계 검사상 즉시 발견된 치명적 구조 오류는 없지만, 내용 검토와 릴리스 승인은 별도입니다.
- `AUDIT_FAIL`: 하나 이상의 필수 검사 실패 또는 누락이 있습니다.
- `FROZEN`: 이 스크립트가 내보내는 상태가 아닙니다.

`review_queue.csv`는 사람이 작성할 검토 대기열입니다. `review_status`, `reviewer`, `rationale`를 근거와 함께 채우되 자동으로 `APPROVED`로 바꾸지 마세요. 수정이 필요하면 원본 덮어쓰기 대신 새 manifest 버전을 만듭니다.
